# SmartFactorBase

The `SmartFactorBase` class is an internal base class for wrapped smart projection factors. It provides their shared measurement and camera workflow.

GTSAM Copyright 2010-2026, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/slam/doc/SmartFactorBase.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass

In [2]:
import gtsam
import numpy as np

from gtsam.symbol_shorthand import X

## Use the base API through a concrete smart factor

`SmartFactorBase` is not constructed directly. We use `SmartProjectionFactorPinholeCameraCal3_S2`, which inherits the base API, and add two image measurements of one landmark. The base class stores the measurement vector and the corresponding camera keys while leaving the landmark implicit.

In [3]:
calibration = gtsam.Cal3_S2(500.0, 500.0, 0.0, 320.0, 240.0)
cameras = [
    gtsam.PinholeCameraCal3_S2(gtsam.Pose3(), calibration),
    gtsam.PinholeCameraCal3_S2(
        gtsam.Pose3(gtsam.Rot3(), np.array([1.0, 0.0, 0.0])),
        calibration,
    ),
]
landmark = np.array([0.0, 0.0, 5.0])
factor = gtsam.SmartProjectionFactorPinholeCameraCal3_S2(
    gtsam.noiseModel.Isotropic.Sigma(2, 1.0)
)
values = gtsam.Values()
for index, camera in enumerate(cameras):
    factor.add(camera.project(landmark), X(index))
    values.insert(X(index), camera)

assert len(factor.measured()) == 2
print("measurements:", factor.measured())
print("camera keys:", factor.keys())

measurements: [array([320., 240.]), array([220., 240.])]
camera keys: [8646911284551352320, 8646911284551352321]


## Recover the cameras and triangulate

`cameras(values)` follows the stored keys and assembles the typed camera set used by the derived factor. With consistent measurements, the factor triangulates the landmark and reports zero total reprojection error.

In [4]:
camera_set = factor.cameras(values)
assert len(camera_set) == 2
assert np.isclose(factor.totalReprojectionError(camera_set), 0.0)
print("camera set type:", type(camera_set).__name__)
print("triangulation valid:", bool(factor.point(values)))

camera set type: CameraSetCal3_S2
triangulation valid: True


## When to use it

`SmartFactorBase` is internal shared machinery for smart projection factors. Application code should construct a derived factor matching the camera type stored in `Values`; the wrapper supplies the corresponding base specialization automatically. The base API is most useful when adding measurements or inspecting how a smart factor resolves its cameras.

Continue with [`SmartProjectionFactor`](SmartProjectionFactor.ipynb) for the nonlinear factor, [`SmartFactorParams`](SmartFactorParams.ipynb) for configuration, and [`JacobianFactorQ`](JacobianFactorQ.ipynb) for one possible linearization result.

## Source

[`SmartFactorBase.h`](../SmartFactorBase.h)


## AI assistance caveat

AI was used to help draft this documentation, and inaccuracies could be present.